# Testing in Machine Learning

## Why ML Testing is Different

Traditional software: deterministic → test exact outputs.
ML software: probabilistic → test statistical properties, thresholds, and behaviors.

### ML Testing Pyramid

```
         ┌──────────────────┐
         │  Model Tests     │  ← Performance, fairness, robustness
        ┌┴──────────────────┴┐
        │  Integration Tests │  ← Pipeline end-to-end
       ┌┴────────────────────┴┐
       │    Data Tests        │  ← Schema, drift, quality
      ┌┴──────────────────────┴┐
      │      Unit Tests        │  ← Features, transforms, utils
      └────────────────────────┘
```

### Types of ML Tests

| Type | What | Tools |
|------|------|-------|
| Unit | Data transforms, feature functions | pytest |
| Data | Schema, distributions, nulls | Great Expectations, Pandera |
| Model | Accuracy thresholds, slices | deepchecks, custom |
| Training | Gradient flow, overfit test | Custom |
| Integration | Full pipeline | pytest + fixtures |
| Infrastructure | API endpoints, latency | httpx, locust |

## 1. Unit Tests with pytest

In [1]:
# feature_engineering.py (the code we want to test)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def clip_outliers(series: pd.Series, n_std: float = 3.0) -> pd.Series:
    """Clip values beyond n standard deviations."""
    mean, std = series.mean(), series.std()
    return series.clip(mean - n_std * std, mean + n_std * std)

def log1p_transform(series: pd.Series) -> pd.Series:
    """Log1p transform requires non-negative input."""
    if (series < 0).any():
        raise ValueError("log1p requires non-negative values")
    return np.log1p(series)

def create_ratio_feature(df: pd.DataFrame, num_col: str, denom_col: str) -> pd.Series:
    """Ratio feature with safe division."""
    return df[num_col] / df[denom_col].replace(0, np.nan)

# ---- tests (normally in test_features.py) ----
import pytest

def test_clip_outliers_removes_extremes():
    s = pd.Series([1, 2, 3, 100])  # 100 is an outlier
    clipped = clip_outliers(s, n_std=1.0)
    assert clipped.max() < 100, "Outlier should be clipped"

def test_clip_outliers_preserves_shape():
    s = pd.Series(np.random.randn(100))
    assert clip_outliers(s).shape == s.shape

def test_log1p_raises_on_negative():
    with pytest.raises(ValueError, match="non-negative"):
        log1p_transform(pd.Series([-1, 0, 1]))

def test_log1p_zero_becomes_zero():
    result = log1p_transform(pd.Series([0.0]))
    assert result.iloc[0] == 0.0

def test_ratio_feature_handles_zero_denom():
    df = pd.DataFrame({"a": [10, 20], "b": [0, 5]})
    result = create_ratio_feature(df, "a", "b")
    assert pd.isna(result.iloc[0]), "Division by zero should produce NaN"
    assert result.iloc[1] == 4.0

# Run tests inline
test_clip_outliers_removes_extremes()
test_clip_outliers_preserves_shape()
test_log1p_raises_on_negative()
test_log1p_zero_becomes_zero()
test_ratio_feature_handles_zero_denom()
print("All unit tests passed!")

All unit tests passed!


In [2]:
# Parametrized tests and fixtures (conftest.py patterns)
import pytest
import numpy as np
import pandas as pd

# Fixture example (normally in conftest.py)
# @pytest.fixture
# def sample_df():
#     np.random.seed(42)
#     return pd.DataFrame({
#         'age': np.random.randint(18, 80, 100),
#         'income': np.random.exponential(50000, 100),
#         'label': np.random.randint(0, 2, 100)
#     })

# Parametrized test
# @pytest.mark.parametrize("n_std,expected_max", [
#     (1.0, True),
#     (2.0, True),
#     (3.0, True),
# ])
# def test_clip_with_different_stds(sample_df, n_std, expected_max):
#     clipped = clip_outliers(sample_df['income'], n_std=n_std)
#     assert (clipped <= sample_df['income'].mean() + n_std * sample_df['income'].std() + 1e-6).all()

print("pytest patterns demonstrated above")
print("Run tests with: pytest tests/ -v --tb=short --cov=src --cov-report=html")

pytest patterns demonstrated above
Run tests with: pytest tests/ -v --tb=short --cov=src --cov-report=html


## 2. Data Testing with Pandera

In [3]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check
import pandas as pd
import numpy as np

# Define schema
input_schema = DataFrameSchema(
    {
        "age": Column(int, Check.in_range(0, 120), nullable=False),
        "income": Column(float, Check.ge(0), nullable=False),
        "education": Column(str, Check.isin(["high_school", "bachelors", "masters", "phd"])),
        "label": Column(int, Check.isin([0, 1]), nullable=False),
    },
    checks=[
        Check(lambda df: df.duplicated().sum() == 0, error="Duplicate rows found"),
    ]
)

# Valid data
valid_df = pd.DataFrame({
    "age": [25, 35, 45],
    "income": [50000.0, 75000.0, 90000.0],
    "education": ["bachelors", "masters", "phd"],
    "label": [0, 1, 0]
})

validated = input_schema.validate(valid_df)
print("Valid data passed schema validation!")

# Invalid data will raise SchemaError
invalid_df = valid_df.copy()
invalid_df.loc[0, "age"] = -5  # invalid age

try:
    input_schema.validate(invalid_df)
except pa.errors.SchemaError as e:
    print(f"Schema error caught: {e.failure_cases}")

Valid data passed schema validation!
Schema error caught:    index  failure_case
0      0            -5


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## 3. Data Testing with Great Expectations

In [4]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd

# Create GX context and validate a batch (Great Expectations 1.x API)
df = pd.DataFrame({
    "age": [25, 35, 45, 28, 52],
    "income": [50000.0, 75000.0, 90000.0, 42000.0, 110000.0],
    "label": [0, 1, 0, 1, 0]
})

context = gx.get_context(mode="ephemeral")
ds = context.data_sources.add_pandas("my_source")
asset = ds.add_dataframe_asset("training_data")
batch_def = asset.add_batch_definition_whole_dataframe("batch_def")
batch = batch_def.get_batch(batch_parameters={"dataframe": df})

expectations = [
    gxe.ExpectColumnValuesToNotBeNull(column="age"),
    gxe.ExpectColumnValuesToBeBetween(column="age", min_value=0, max_value=120),
    gxe.ExpectColumnValuesToNotBeNull(column="income"),
    gxe.ExpectColumnValuesToBeBetween(column="income", min_value=0),
    gxe.ExpectColumnValuesToBeInSet(column="label", value_set=[0, 1]),
    gxe.ExpectTableRowCountToBeBetween(min_value=100, max_value=None),  # this will fail
]

all_success = True
for exp in expectations:
    res = batch.validate(exp)
    all_success = all_success and res.success
    status = "PASS" if res.success else "FAIL"
    print(f"  [{status}] {type(exp).__name__}")
print(f"Success: {all_success}")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

  [PASS] ExpectColumnValuesToNotBeNull


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

  [PASS] ExpectColumnValuesToBeBetween


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

  [PASS] ExpectColumnValuesToBeBetween


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

  [PASS] ExpectColumnValuesToBeInSet


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

  [FAIL] ExpectTableRowCountToBeBetween
Success: False


## 4. Model Output Testing

In [5]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# 1. Threshold-based tests
def test_accuracy_threshold(y_true, y_pred, min_acc=0.90):
    acc = accuracy_score(y_true, y_pred)
    assert acc >= min_acc, f"Accuracy {acc:.3f} below threshold {min_acc}"
    return acc

def test_auc_threshold(y_true, y_prob, min_auc=0.95):
    auc = roc_auc_score(y_true, y_prob)
    assert auc >= min_auc, f"AUC {auc:.3f} below threshold {min_auc}"
    return auc

# 2. Output shape / type tests
def test_output_shapes(model, X):
    preds = model.predict(X)
    probs = model.predict_proba(X)
    assert preds.shape == (len(X),), "Prediction shape mismatch"
    assert probs.shape == (len(X), 2), "Probability shape mismatch"
    assert np.allclose(probs.sum(axis=1), 1.0), "Probabilities must sum to 1"

# 3. Slice-based testing (fairness / subgroup performance)
def test_slice_performance(model, X_test, y_test, feature, threshold, min_acc=0.85):
    mask = X_test[feature] > threshold
    for slice_name, slice_mask in [("above", mask), ("below", ~mask)]:
        if slice_mask.sum() < 10:
            continue
        acc = accuracy_score(y_test[slice_mask], model.predict(X_test[slice_mask]))
        assert acc >= min_acc, f"Slice '{slice_name}' accuracy {acc:.3f} below {min_acc}"
        print(f"  Slice '{feature} {slice_name} {threshold}': acc={acc:.3f} n={slice_mask.sum()}")

# 4. Metamorphic test: negating features shouldn't flip all predictions
def test_metamorphic(model, X_test, max_flip_rate=0.5):
    orig_preds = model.predict(X_test)
    perturbed = X_test + np.random.normal(0, 0.01, X_test.shape)
    perturbed_preds = model.predict(perturbed)
    flip_rate = (orig_preds != perturbed_preds).mean()
    assert flip_rate < max_flip_rate, f"Too many flips ({flip_rate:.1%}) from tiny perturbations"
    return flip_rate

# Run all tests
acc = test_accuracy_threshold(y_test, y_pred)
print(f"Accuracy: {acc:.3f} ✓")

auc = test_auc_threshold(y_test, y_prob)
print(f"AUC: {auc:.3f} ✓")

test_output_shapes(model, X_test)
print("Output shapes ✓")

print("Slice tests:")
test_slice_performance(model, X_test, y_test, feature=X_test.columns[0], threshold=X_test.iloc[:, 0].median())

flip_rate = test_metamorphic(model, X_test.values)
print(f"Metamorphic test flip rate: {flip_rate:.1%} ✓")

Accuracy: 0.958 ✓
AUC: 0.995 ✓
Output shapes ✓
Slice tests:
  Slice 'mean radius above 13.59': acc=0.930 n=71


  Slice 'mean radius below 13.59': acc=0.986 n=72
Metamorphic test flip rate: 0.7% ✓


## 5. Training Code Tests

### Gradient Check

Numerical gradient approximation:
$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + \epsilon e_i) - L(\theta - \epsilon e_i)}{2\epsilon}$$

Compare analytical gradient to numerical gradient they should match to within $$10^{-5}$$.

In [6]:
import torch
import torch.nn as nn
import numpy as np

# Simple model for testing
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        return self.fc(x)

model = TinyNet()
loss_fn = nn.CrossEntropyLoss()
x = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

# Test 1: Loss decreases when training
def test_loss_decreases():
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    losses = []
    for _ in range(20):
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    assert losses[-1] < losses[0], f"Loss did not decrease: {losses[0]:.4f} → {losses[-1]:.4f}"
    return losses

# Test 2: Model can overfit a tiny dataset (sanity check)
def test_model_can_overfit():
    tiny_model = TinyNet()
    opt = torch.optim.Adam(tiny_model.parameters(), lr=0.1)
    x_tiny = torch.randn(4, 4)
    y_tiny = torch.tensor([0, 1, 0, 1])
    for _ in range(200):
        opt.zero_grad()
        loss = loss_fn(tiny_model(x_tiny), y_tiny)
        loss.backward()
        opt.step()
    final_acc = (tiny_model(x_tiny).argmax(dim=1) == y_tiny).float().mean()
    assert final_acc == 1.0, f"Model should overfit tiny dataset, got acc={final_acc}"

# Test 3: Output shape
def test_output_shape():
    out = model(x)
    assert out.shape == (8, 2), f"Expected (8,2) got {out.shape}"

# Test 4: No NaN in gradients
def test_no_nan_gradients():
    out = model(torch.randn(4, 4))
    loss = loss_fn(out, torch.randint(0, 2, (4,)))
    loss.backward()
    for name, param in model.named_parameters():
        assert not torch.isnan(param.grad).any(), f"NaN gradient in {name}"

test_output_shape()
print("Output shape test passed ✓")

losses = test_loss_decreases()
print(f"Loss decrease test passed ✓ ({losses[0]:.4f} → {losses[-1]:.4f})")

test_model_can_overfit()
print("Overfit test passed ✓")

test_no_nan_gradients()
print("No NaN gradients ✓")

Output shape test passed ✓


Loss decrease test passed ✓ (1.1576 → 1.0298)


Overfit test passed ✓
No NaN gradients ✓


## 6. API Testing with FastAPI TestClient

In [7]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

# Build a minimal ML API
X, y = load_breast_cancer(return_X_y=True)
_model = RandomForestClassifier(n_estimators=10, random_state=42).fit(X, y)

app = FastAPI()

class PredictRequest(BaseModel):
    features: list[float]

class PredictResponse(BaseModel):
    prediction: int
    probability: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    x = np.array(req.features).reshape(1, -1)
    pred = int(_model.predict(x)[0])
    prob = float(_model.predict_proba(x)[0][pred])
    return PredictResponse(prediction=pred, probability=prob)

# Test the API
client = TestClient(app)

def test_health_endpoint():
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"

def test_predict_valid_input():
    features = X[0].tolist()
    r = client.post("/predict", json={"features": features})
    assert r.status_code == 200
    data = r.json()
    assert data["prediction"] in [0, 1]
    assert 0.0 <= data["probability"] <= 1.0

def test_predict_wrong_feature_count():
    r = client.post("/predict", json={"features": [1.0, 2.0]})  # wrong number
    assert r.status_code == 422 or r.status_code == 500

def test_predict_response_time():
    import time
    features = X[0].tolist()
    start = time.time()
    client.post("/predict", json={"features": features})
    elapsed = time.time() - start
    assert elapsed < 1.0, f"Response time {elapsed:.2f}s exceeds 1s threshold"

test_health_endpoint()
print("Health endpoint ✓")
test_predict_valid_input()
print("Valid prediction ✓")
test_predict_response_time()
print("Response time ✓")

Health endpoint ✓
Valid prediction ✓
Response time ✓


## 7. Continuous Testing in CI

```yaml
# .github/workflows/ml-tests.yml
name: ML Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: pip install -r requirements.txt
      - name: Unit tests
        run: pytest tests/unit -v
      - name: Data validation
        run: python scripts/validate_data.py
      - name: Model tests
        run: pytest tests/model -v --tb=short
      - name: Coverage
        run: pytest --cov=src --cov-fail-under=80
```

## Additional Learning Resources

### Documentation
- **Great Expectations**: https://docs.greatexpectations.io/
- **Pandera**: https://pandera.readthedocs.io/
- **deepchecks**: https://docs.deepchecks.com/
- **pytest**: https://docs.pytest.org/

### Papers & Articles
- **"Testing Machine Learning Systems: Code, Data and Models"** Breck et al. (Google): https://research.google/pubs/pub46555/
- **"What's your ML test score?"**: https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/45742.pdf
- **Effective Testing for ML**: https://eugeneyan.com/writing/testing-ml/
- **Beyond Accuracy: Behavioral Testing of NLP models with CheckList**: https://arxiv.org/abs/2005.04118

### Courses
- **Made With ML Testing**: https://madewithml.com/courses/mlops/testing/
- **Full Stack Deep Learning Testing**: https://fullstackdeeplearning.com/course/2022/lecture-4-data-management/